# Predict Using the pretrained model

In [ ]:
from transformers import AutoModelForTokenClassification, AutoTokenizer
import torch
from torch.nn.functional import softmax

In [ ]:
# Load model, tokenizer, and labels
model_name = "emanjavacas/GysBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# Define your labels (NER example - adjust as needed)
label_list = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]
num_labels = len(label_list)
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in id2label.items()}

# Load the token classification model
model = AutoModelForTokenClassification.from_pretrained(
    model_name, 
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)



In [ ]:
# Set to evaluation mode
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

In [ ]:
def predict_tokens(text):
    """Predict token labels for input text"""
    inputs = tokenizer(
        text, 
        return_tensors="pt", 
        truncation=True, 
        padding=True, 
        max_length=512
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=-1)[0]
        probabilities = softmax(outputs.logits[0], dim=-1)
    
    # Decode tokens and predictions
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    results = []
    
    for token, pred_id, probs in zip(tokens, predictions, probabilities):
        pred_label = id2label[pred_id.item()]
        max_prob = torch.max(probs).item()
        results.append((token, pred_label, max_prob))
    
    return results

# Test inference
# text = "Jan de Vries werkt bij de Universiteit van Amsterdam in Nederland."
text = "FRANKRIJK.</title>\n\t\n<p>&#226;&#128;&#148;. De Parijsche Monileur meldt, dat de regering inslructi&#195;&#171;n naar Mexico gezouden heeft tot intrekking van de maatregelen betreffende het in beslag nemen der goederen van hen die tegen Frankrijk de wapenen voeren, en betreffende het verbod van gelduitvoer. Ofschoon de toestand verbeterd is, bestaan in Mexico nog gewapende benden die uit zekere havens hulp eu bijstand ontvangen. Ten einde nu de verstrooijing dier bendeu te bespoedigen, sal de Fransche admiraal, te beginnen met 25 Augustus, de kv.s*. blokkeren , van de Lagunen op lien mijlen ten zuiden van Malaraoras tot en met Camp&#195;&#170;che.</p>\n\t\n<p>La France zegt dat de aanvaarding der keizerskroon in Mexico door den aartshertog Maximiliaan uiet twijfelachtig is. Het blad meent te weten dat reeds onderhandelingen ziju aangeknoopt met Engeland , om de adhesie dier mogendheid te erlangen, en dat de raad van regentschap, die (hans met htt bestuur in Mexico belast is , gedurende etn jaar zal blijven fungeren, ten einde het land te organiseren; doch dat het berigt vau de toestemming des aartshertogs waarschijnlijk reeds in de maand November a. s. vaar Mexico zal worden gezonden.</p>\n\t\n<p>De Osldeutsche Post acht het echter voor eene uitgemaakte zaak dat de aartshertog Maximiliaan de Mexicaansche kroon niet zal aannemen. Ook de General Correspondenz zegt dat al de mededeelingen over de Mexicaansche zaak voorbarig zijn ; dat daarin nog hoegenaamd geeu definitief besluit is genomen; dat de uit Mexico naar Weenen vertrokken deputatie niet in den naam van het gansclie land kan spreken , en dut de verdere loop der gebeurtenissen moet worden afgewacht, alvorens hier een besluit kan worden genomen.</p>\n\t\n<p>Dezer dagen is te Bordeaux in eeue distilleerderij ecu brand ontstaan in een vat alcohol, die binneu korten tijd een verlies van li a 3 millioen fr. heeft veroorzaakt.</p>"
predictions = predict_tokens(text)

In [ ]:
# Pretty print results
print("Token Classification Results:")
print("=" * 60)
for i, (token, label, confidence) in enumerate(predictions):
    if token not in ['[CLS]', '[SEP]', '[PAD]']:
        print(f"{token:<15} {label:<10} {confidence:.3f}")

In [ ]:
texts = [
    "Jan werkt bij KPN in Amsterdam.",
    "Marie Curie won de Nobelprijs."
]
for text in texts:
    print(f"\n--- {text} ---")
    preds = predict_tokens(text)
    for token, label, conf in preds:
        if token not in ['[CLS]', '[SEP]', '[PAD]']:
            print(f"{token:<15} {label:<10} {conf:.3f}")
